In [15]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.genmod.generalized_linear_model import GLM
from statsmodels.genmod.generalized_linear_model import GLMResultsWrapper
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression



In [12]:
import sys
import os
from pathlib import Path

path = Path(os.path.abspath(""))
base_path = path.parent.absolute()
telco_path = os.path.join(str(base_path), '1_dataset', 'telco', 'telco_churn_data.csv')

In [13]:


class Statsmodels2SklearnInterface():
    pass
    # implements methods from the sklearn interface


class SklearnLikeBase:

    def predict(self, X, threshold=0.5):
        """
        Predict binary outcomes (0 or 1) based on a threshold.
        """
        probabilities = self.predict_prob(X)
        return (probabilities >= threshold).astype(int)

    def summary(self):
        """
        Returns the summary of the fitted GLM model.
        """
        if self.results is None:
            raise ValueError("Model is not fitted yet. Call `fit` before `summary`.")
        return self.results.summary()

    def get_params(self, deep=True):
        """
        Returns the parameters of the estimator (mimics sklearn API).
        """
        return {"family": self.family}

    def set_params(self, **params):
        """
        Sets the parameters of the estimator (mimics sklearn API).
        """
        for key, value in params.items():
            setattr(self, key, value)
        return self


class OldSklearnLikeGLM(SklearnLikeBase):
    def __init__(self, family=None):
        """
        Initializes the GLM model.
        :param family: A statsmodels family object. Default is sm.families.Binomial for logistic regression.
        """
        self.family = family if family else sm.families.Binomial()
        self.model = None
        self.results = None
        self.has_constant = False # Tracks whether a constant was added during `fit`
        self._X_columns = None # Store original column names
        self._formula_columns = None # Store column names used in the formula
        self._inv_formula_columns = None

    def fit(self, X, y):
        """
        Fits the GLM model with a constant term.
        """
        # Ensure X is a DataFrame
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)

        # Store original column names
        self._X_columns = X.columns.tolist()

        # Add constant term
        X = sm.add_constant(X)
        self.has_constant = True

        # Combine X and y into a single DataFrame for formula-based modeling
        data = X.copy()
        data['y'] = y

        # Create safe column names for the formula
        self._formula_columns = {
            col: f"col_{i}" for i, col in enumerate(X.columns)
        }

        # Broken below
        # self._inv_formula_columns = {v:k for k,v in model._formula_columns.items()}

        # data.rename(columns=self._formula_columns, inplace=True)

        # # Construct the formula
        # formula = "y ~ " + " + ".join(self._formula_columns.values())

        # # Fit GLM model
        # self.model = smf.glm(formula=formula, data=data, family=Binomial(link=logit())) #family=self.family)
        # self.results = self.model.fit(maxiter=1000)

        return self

    def predict_prob(self, X):
        """
        Predict probabilities using the fitted model.
        """
        if self.results is None:
            raise ValueError("Model is not fitted yet. Call `fit` before `predict`.")

        # Ensure X is a DataFrame with the same columns as training data
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X, columns=self._X_columns)

        if self.has_constant:
            X = sm.add_constant(X, has_constant='add')

        # Rename columns to match those used during fitting
        X.rename(columns=self._formula_columns, inplace=True)
        return self.results.predict(X)

    # def summary(self):
    #     """
    #     Returns the summary of the fitted GLM model.
    #     """
    #     if self.results is None:
    #         raise ValueError("Model is not fitted yet. Call `fit` before `summary`.")
    #     summary = self.results.summary()
    #     data.rename(columns=self._formula_columns, inplace=True)
    #     model._inv_formula_columns

    def summary(self, data):
        """
        Returns the summary of the fitted GLM model.
        """
        if self.results is None:
            raise ValueError("Model is not fitted yet. Call `fit` before `summary`.")
        summary = self.results.summary()
        data.rename(columns=self._formula_columns, inplace=True)
        self.model._inv_formula_columns


class CustomGLMResults(GLMResultsWrapper):
    """Custom results wrapper to clean up parameter names in summary"""
    def summary(self, *args, **kwargs):
        summary = super().summary(*args, **kwargs)
        # Get the cleaned parameter names (remove Q('...'))
        param_names = [name.replace("Q('", "").replace("')", "") 
                      if name != "const" else "Intercept"
                      for name in self.model.exog_names]
        # Replace in summary
        for i, name in enumerate(param_names):
            summary.tables[1].data[i][0] = name
        return summary


class CustomGLM(sm.GLM):
    """Custom GLM class that uses our custom results wrapper"""
    _results_class = CustomGLMResults


class SklearnLikeGLM(SklearnLikeBase):

    # TODO(1.8): Remove this attribute
    _estimator_type = "classifier"


    def __init__(self, family=None):
        """
        Initializes the GLM model.
        :param family: A statsmodels family object. Default is sm.families.Binomial for logistic regression.
        """
        self.family = family if family else sm.families.Binomial()
        self.model = None
        self.results = None
        self.has_constant = False  # Tracks whether a constant was added during fit
        self._X_columns = None  # Store original column names
        self.is_fitted = False
        self.classes_ = np.array([1, 0])

    def __sklearn_tags__(self):
        tags = super().__sklearn_tags__()
        tags.estimator_type = "classifier"
        # tags.classifier_tags = ClassifierTags()
        tags.target_tags.required = True
        return tags

    def fit(self, X, y):
        """
        Fits the GLM model with a constant term.
        """
        # Ensure X is a DataFrame
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)
        
        # Store original column names
        self._X_columns = X.columns.tolist()
        
        # Add constant term with the name 'Intercept'
        X = pd.DataFrame(sm.add_constant(X, has_constant='add'))
        X.rename(columns={'const': 'Intercept'}, inplace=True)
        self.has_constant = True
        
        # Create the model using the DataFrame with proper column names
        self.model = GLM(y, X, family=self.family)
        # self.model = smf.glm(formula=formula, data=data, family=Binomial(link=logit())) #family=self.family)
        self.results = self.model.fit(maxiter=1000)
        self.is_fitted = True
        return self

    def predict_proba(self, X):
        # return self.predict_prob(X)
        return np.stack([
            self.predict_prob(X),
            np.array(1) - self.predict_prob(X),
        ], axis=1)

    def predict(self, X):
        y_prob = self.predict_prob(X)
        y_pred = (y_prob >= 0.5).astype(int).values
        return y_pred

    def predict_prob(self, X):
        """
        Predict probabilities using the fitted model.
        """
        if self.results is None:
            raise ValueError("Model is not fitted yet. Call fit before predict.")
        
        # Ensure X is a DataFrame with the same columns as training data
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X, columns=self._X_columns)
            
        # Add constant if it was used during fitting
        if self.has_constant:
            X = sm.add_constant(X, has_constant='add')
            X = pd.DataFrame(X)
            X.rename(columns={'const': 'Intercept'}, inplace=True)
            
        return self.results.predict(X)

    def __sklearn_is_fitted__(self):
        return self.is_fitted


class SklearnLikeLogit(SklearnLikeBase):
    def __init__(self):
        self.model = None
        self.results = None
        self.has_constant = False  # Tracks whether a constant was added during `fit`
        self._X_columns = None  # Store original column names

    def fit(self, X, y):
        """
        Fits the logistic regression model with a constant term.
        """
        # if self.results is None:
        #     raise ValueError("Model is not fitted yet. Call `fit` before `predict`.")

        # Ensure X is a DataFrame
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)

        # Store original column names
        self._X_columns = X.columns.tolist()

        # Add constant term
        X = sm.add_constant(X, has_constant='add')
        self.has_constant = True

        self.model = sm.Logit(y, X)
        self.results = self.model.fit() #disp=False # method='newton', maxiter=100, disp=True
        # self.results = self.model.fit_regularized(start_params=None, method='l1', maxiter='defined_by_method', full_output=1, disp=1, callback=None, alpha=0.01, trim_mode='size', auto_trim_tol=0.01, size_trim_tol=0.001, qc_tol=0.03,)

        return self

    def predict_prob(self, X):
        """
        Predict probabilities using the fitted model.
        """
        if self.results is None:
            raise ValueError("Model is not fitted yet. Call `fit` before `predict`.")

        # Ensure X is a DataFrame with the same columns as training data
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X, columns=self._X_columns)

        # Add constant term
        if self.has_constant:
            X = sm.add_constant(X, has_constant='add')

        return self.results.predict(X)


# classifier_logreg = SklearnLikeGLM()
# classifier_logreg.fit(x_train_df, y_train)
# classifier_logreg.predict(x_train_df)
# classifier_logreg.predict_proba(x_train_df)

# RocCurveDisplay.from_estimator(classifier_logreg, x_test, y_test)

In [24]:
df = pd.read_csv(telco_path)

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

def augment_dataset_datetime_random(df, start_date, end_date):
    _df = df.copy()
    n_rows = len(_df)
    random_days = np.random.randint(0, (end_date - start_date).days + 1, size=n_rows)
    _df['Date'] = start_date + pd.to_timedelta(random_days, unit='D')
    return _df

def augment_dataset_datetime_monthend(df, start_date, end_date):
    _df = df.copy()
    n_rows = len(_df)
    month_end_dates = pd.date_range(start=start_date, end=end_date, freq='M').to_list()
    if pd.to_datetime(end_date).day != pd.to_datetime(end_date).daysinmonth:
        if pd.to_datetime(end_date) > month_end_dates[-1]:
            month_end_dates.append(pd.to_datetime(end_date))
    random_month_ends = np.random.choice(month_end_dates, size=n_rows, replace=True)
    _df['Date'] = random_month_ends
    return _df



# Datetime Augmentation
start_date = pd.to_datetime('2023-01-01')
end_date = pd.to_datetime('2023-07-31')




# df = augment_dataset_location_random(df, location_list, weights=weights)
# df = augment_dataset_location_cluster(df, features_to_cluster, n_clusters)
# df['Location'] = augment_dataset_location_probabilistic(df, 'MonthlyCharges', location_config)
df = augment_dataset_datetime_monthend(df, start_date, end_date)
# df = expand_customer_history()

features_col = [
    'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
    'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges'
]
target_col = 'Churn'

df.dropna(inplace=True)

def object_to_int(dataframe_series):
    if dataframe_series.dtype=='object':
        dataframe_series = LabelEncoder().fit_transform(dataframe_series)
    return dataframe_series


def object_to_int2(df, feature_list):
    for feat in feature_list:
        if df[feat].dtype == 'object':
            df[feat] = LabelEncoder().fit_transform(df[feat])
    return df


# df = df.apply(lambda x: object_to_int2(x))
df = object_to_int2(df, features_col + [target_col])

df.head()

X_df = df.drop(columns=['Churn'])
y_df = df[['Churn']]



X_train_df, X_test_df, y_train_df, y_test_df = train_test_split(X_df, y_df, test_size=0.30, random_state=40) # , stratify=y


C:\Users\douglas.sgrott_indic\AppData\Local\Temp\ipykernel_11016\3566164500.py:15: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  month_end_dates = pd.date_range(start=start_date, end=end_date, freq='M').to_list()


In [26]:
sklearn_model = LogisticRegression()
sklearn_model.fit(X_train_df[features_col], y_train_df[target_col])

c:\Users\douglas.sgrott_indic\miniconda3\envs\tsfresh_venv\lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [30]:
print(sklearn_model.intercept_)
print(sklearn_model.coef_)

[-1.45750923]
[[ 0.22198996  0.2051266  -0.39124896 -0.03406179 -0.26578399 -0.13475361
  -0.05593571 -0.20031829 -0.72358988  0.43803947  0.07237907  0.02454324]]


In [27]:
classifier_logreg = SklearnLikeGLM()
classifier_logreg.fit(X_train_df[features_col], y_train_df[target_col])


In [29]:
classifier_logreg.results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:                  Churn   No. Observations:                 4922
Model:                            GLM   Df Residuals:                     4909
Model Family:                Binomial   Df Model:                           12
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -2052.7
Date:                Mon, 11 Aug 2025   Deviance:                       4105.4
Time:                        16:11:20   Pearson chi2:                 5.11e+03
No. Iterations:                     6   Pseudo R-squ. (CS):             0.2790
Covariance Type:            nonrobust                                         
====================================================================================
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept           -1.4801      0.160     -9.237      0.000      -1.794      -1.166
SeniorCitizen        0.2402      0.101      2.372      0.018       0.042       0.439
Partner              0.1987      0.093      2.144      0.032       0.017       0.380
Dependents          -0.3599      0.107     -3.356      0.001      -0.570      -0.150
tenure              -0.0340      0.003    -12.392      0.000      -0.039      -0.029
OnlineSecurity      -0.2683      0.049     -5.468      0.000      -0.364      -0.172
OnlineBackup        -0.1322      0.045     -2.938      0.003      -0.220      -0.044
DeviceProtection    -0.0510      0.046     -1.103      0.270      -0.142       0.040
TechSupport         -0.1987      0.050     -3.971      0.000      -0.297      -0.101
Contract            -0.7348      0.092     -8.005      0.000      -0.915      -0.555
PaperlessBilling     0.4472      0.089      5.052      0.000       0.274       0.621
PaymentMethod        0.0765      0.042      1.821      0.069      -0.006       0.159
MonthlyCharges       0.0245      0.002     14.064      0.000       0.021       0.028
====================================================================================
"""